# WAHIS Exploratory Data Analysis — Poultry Disease Early Warning System for Nigeria

**Purpose:** understand the structure, reporting coverage, distributions, temporal behaviour, administrative variation and relationships in the WOAH/WAHIS quantitative poultry-disease export before model development.

This notebook complements the interactive Streamlit dashboard. It uses the raw source export and the repository's conservative cleaning and grouping helpers. It does not retrain a model.

**Important interpretation rules**
- `New outbreaks` records reported new outbreak counts, not the number of infected birds.
- A dash or blank is unavailable information, not zero.
- Repeated source rows are not automatically independent outbreak events.
- The analytical comparison grain is year × semester × administrative division × disease.
- Historical counts and geographic rankings describe reported surveillance activity, not true disease prevalence or causal risk.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.cleaning import clean_wahis
from src.data.panel import build_observed_panel

SOURCE_PATH = PROJECT_ROOT / 'data' / 'raw' / 'Quantitative data 2026-10-01.csv'
assert SOURCE_PATH.exists(), f'Raw WAHIS file not found: {SOURCE_PATH}'
raw_source = pd.read_csv(SOURCE_PATH, dtype='string')
df = clean_wahis(raw_source)
print(f'Source shape: {raw_source.shape[0]:,} rows × {raw_source.shape[1]:,} columns')
print(f'Cleaned shape: {df.shape[0]:,} rows × {df.shape[1]:,} columns')


## 1. Source structure and data types
Start by checking the fields, types, period coverage and the first rows. This confirms what the source actually contains before drawing conclusions.

In [ ]:
display(df.head(10))
print('Column types:')
display(df.dtypes.astype(str).rename('dtype').to_frame())
print('Reporting years:', int(df.year.min()), 'to', int(df.year.max()))
print('Disease categories:', df.disease.nunique(dropna=True))
print('Administrative labels:', df.administrative_division.nunique(dropna=True))


## 2. Completeness of the outbreak-count field
The WAHIS source contains 1,180 rows. Count how many have a numeric `new_outbreaks` value. Keep unavailable values separate from observed zeroes.

In [ ]:
outbreak_availability = (
    df.assign(outbreak_count_status=np.where(df.new_outbreaks.notna(), 'Observed numeric count', 'Unavailable'))
      .groupby('outbreak_count_status').size().rename('source_rows').reset_index()
)
display(outbreak_availability)
fig = px.bar(outbreak_availability, x='outbreak_count_status', y='source_rows', color='outbreak_count_status',
             title='Availability of numeric new-outbreak counts', text='source_rows',
             color_discrete_map={'Observed numeric count':'#159c91','Unavailable':'#e8aa4d'})
fig.update_layout(showlegend=False, yaxis_title='Source rows', xaxis_title='')
fig.show()


## 3. Disease coverage
Compare how much numeric outbreak-count evidence is available for each disease. The observed count total is shown for context, but it is not a population-adjusted disease burden measure.

In [ ]:
disease_coverage = (
    df.groupby('disease', dropna=False)
      .agg(source_rows=('disease','size'), observed_rows=('new_outbreaks','count'),
           observed_reported_count_sum=('new_outbreaks', lambda x: x.sum(min_count=1)))
      .reset_index().sort_values('observed_rows', ascending=False)
)
disease_coverage['unavailable_rows'] = disease_coverage.source_rows - disease_coverage.observed_rows
display(disease_coverage)
fig = px.bar(disease_coverage, x='disease', y=['observed_rows','unavailable_rows'], barmode='stack',
             title='Numeric outbreak-count availability by disease',
             labels={'value':'Source rows','disease':'Disease','variable':'Availability'})
fig.update_layout(xaxis_tickangle=-25)
fig.show()


## 4. Build the unique area–disease–semester panel
Multiple source rows can represent companion quantitative fields for the same reporting group. The project helper retains a measure only if its non-missing values are unambiguous. It does not add repeated values automatically.

In [ ]:
panel = build_observed_panel(df)
panel['year'] = pd.to_numeric(panel.year, errors='coerce').astype('Int64')
panel['semester_half'] = panel.semester.astype('string').str.extract(r'^(Jan-Jun|Jul-Dec)', expand=False)
panel['semester_order'] = panel.semester_half.map({'Jan-Jun':1, 'Jul-Dec':2}).astype('Int64')
panel['temporal_period'] = panel.year.astype('Float64') * 2 + (panel.semester_order.astype('Float64') - 1)
panel['period_label'] = panel.semester.astype('string')
panel['admin_display'] = panel.administrative_division.astype('string').replace({'Nassarawa':'Nasarawa'})
panel['is_state_level'] = panel.administrative_division.notna() & panel.administrative_division.ne('Nigeria')

quality = pd.DataFrame({
 'measure':['Raw source rows','Unique area–disease–semester groups','Groups with observed outbreak count',
            'Groups with unavailable outbreak count','Groups with multiple source rows',
            'Groups with conflicting outbreak counts','Groups with conflicting case values'],
 'value':[len(df), len(panel), int(panel.new_outbreaks.notna().sum()), int(panel.new_outbreaks.isna().sum()),
          int(panel.source_record_count.gt(1).sum()), int(panel.new_outbreaks_ambiguous.sum()), int(panel.cases_ambiguous.sum())]
})
display(quality)


## 5. Distribution of observed outbreak counts
The target values are right-skewed: many groups have small counts and relatively few groups have large counts. Use both a histogram and a box plot to inspect the distribution and outliers.

In [ ]:
observed_counts = panel.new_outbreaks.dropna()
display(observed_counts.describe(percentiles=[.25,.5,.75,.9,.95,.99]).to_frame('new_outbreaks'))
fig = px.histogram(panel.dropna(subset=['new_outbreaks']), x='new_outbreaks', nbins=30,
                   title='Distribution of observed new-outbreak counts', color_discrete_sequence=['#159c91'])
fig.update_layout(xaxis_title='Observed new-outbreak count', yaxis_title='Number of groups')
fig.show()
fig = px.box(panel.dropna(subset=['new_outbreaks']), y='new_outbreaks', points='outliers',
             title='Spread and outliers in observed new-outbreak counts', color_discrete_sequence=['#e8aa4d'])
fig.show()


## 6. Temporal reporting patterns
Aggregate observed counts only, after grouping to the unique analytical grain. Do not fill periods with no numeric report as zero.

In [ ]:
state_panel = panel.loc[panel.is_state_level & panel.new_outbreaks.notna()].copy()
period_trend = (state_panel.groupby(['temporal_period','period_label'], as_index=False)
                .agg(reported_count_sum=('new_outbreaks','sum'), observed_groups=('new_outbreaks','count'))
                .sort_values('temporal_period'))
display(period_trend)
fig = px.line(period_trend, x='period_label', y='reported_count_sum', markers=True,
             title='Observed reported outbreak counts by six-month period',
             labels={'period_label':'Reporting semester','reported_count_sum':'Sum of observed reported counts'},
             color_discrete_sequence=['#159c91'])
fig.show()


## 7. Disease-specific trends
Compare the time pattern for each disease. A missing period means no numeric observation was available for that disease-period combination; it is not evidence of zero outbreaks.

In [ ]:
disease_period = (state_panel.groupby(['temporal_period','period_label','disease'], as_index=False)
                  .agg(reported_count_sum=('new_outbreaks','sum'), observed_groups=('new_outbreaks','count'))
                  .sort_values('temporal_period'))
fig = px.line(disease_period, x='period_label', y='reported_count_sum', color='disease', markers=True,
              title='Reported counts by disease and semester',
              labels={'period_label':'Reporting semester','reported_count_sum':'Sum of observed reported counts'})
fig.update_layout(xaxis_tickangle=-25)
fig.show()


## 8. Administrative variation
State-level comparisons exclude the country-level Nigeria aggregate. Standardise the Nassarawa/Nasarawa spelling variant for comparison. Do not interpret totals as disease risk without poultry-population and surveillance-coverage denominators.

In [ ]:
area_summary = (state_panel.groupby('admin_display', as_index=False)
                .agg(reported_count_sum=('new_outbreaks','sum'), observed_groups=('new_outbreaks','count'),
                     diseases_reported=('disease','nunique')))
area_summary = area_summary.sort_values('reported_count_sum', ascending=False)
display(area_summary.head(15))
fig = px.bar(area_summary.head(15).sort_values('reported_count_sum'), x='reported_count_sum', y='admin_display',
             orientation='h', color='observed_groups',
             title='Administrative areas with the largest observed reported counts',
             labels={'reported_count_sum':'Sum of observed reported counts','admin_display':'Administrative area',
                     'observed_groups':'Groups with observed counts'}, color_continuous_scale='Tealgrn')
fig.show()


## 9. Relationships between outbreak counts and other reported measures
Use pairwise complete observations. Correlations describe association only; they do not demonstrate a biological or causal relationship. The denominator can differ by measure because reporting completeness differs.

In [ ]:
relationship_measures = ['cases','susceptible','deaths','vaccinated','killed_and_disposed_of','slaughtered']
relationship_rows = []
for measure in relationship_measures:
    pair = state_panel[['new_outbreaks', measure]].dropna()
    relationship_rows.append({'measure':measure, 'matched_groups':len(pair),
                              'spearman_rho':pair.corr(method='spearman').iloc[0,1] if len(pair)>=3 else np.nan})
relationship_summary = pd.DataFrame(relationship_rows).sort_values('matched_groups', ascending=False)
display(relationship_summary)
measure = 'cases'
pair = state_panel[['new_outbreaks',measure,'admin_display','disease','period_label']].dropna()
fig = px.scatter(pair, x=measure, y='new_outbreaks', color='disease', hover_data=['admin_display','period_label'],
                 title='Observed outbreak count compared with reported cases',
                 labels={measure:'Reported cases','new_outbreaks':'Observed new-outbreak count'})
fig.show()


## 10. Historical outbreak features and target relationship
This section compares the observed target with available lag/history features in the saved modelling table, if that artifact is present. It is useful for understanding the model's signal, but it is not a substitute for out-of-sample testing.

In [ ]:
evaluation_path = PROJECT_ROOT / 'artifacts' / 'evaluation' / 'model_evaluation_dataset.csv'
if evaluation_path.exists():
    model_data = pd.read_csv(evaluation_path)
    history_cols = ['previous_semester_outbreak_count','two_semesters_prior_outbreak_count',
                    'outbreak_history_mean_3','outbreak_history_max_3']
    rows=[]
    for col in history_cols:
        pair=model_data[[col,'target_outbreak_count']].dropna()
        rows.append({'feature':col,'matched_rows':len(pair),
                     'spearman_rho':pair.corr(method='spearman').iloc[0,1] if len(pair)>=3 else np.nan})
    history_relationships=pd.DataFrame(rows).sort_values('spearman_rho',ascending=False)
    display(history_relationships)
else:
    print('Model evaluation dataset not found. Run the modelling workflow before this optional section.')


## 11. Key findings and interpretation
The following conclusions should be checked against the latest source extract and retained when presenting the EDA:

- The source has a long historical span, but numeric new-outbreak coverage is concentrated in the 2020–2025 period.
- Outbreak-count values are right-skewed, with many small observations and a small number of large counts.
- Newcastle disease has the largest number of numeric outbreak-count groups in the current extract.
- Administrative totals describe observed reported counts, not population-adjusted risk.
- Historical outbreak variables show stronger descriptive relationships with the current target than the selected climate features in the model evaluation dataset.
- Missing values must remain distinct from confirmed zero counts.
- Multiple source rows exist for many analytical groups. The group-level transformation must remain transparent and measure-specific.

## 12. Limitations and next analytical steps
1. Obtain additional numeric surveillance periods and clarify the completeness of zero-count reporting.
2. Confirm the appropriate forecasting grain and target definition with an animal-health domain expert.
3. Build a complete temporal grid so lag features represent consecutive six-month periods rather than previous available rows.
4. Use only predictors available before the forecast semester.
5. Compare historical baselines and count models on identical chronological folds.
6. Reserve a later untouched reporting period for final evaluation when new data are available.
7. Add denominators such as poultry population, reporting completeness and surveillance effort where reliable data can be obtained.

**This is an exploratory description of reported surveillance data. It is not a clinical diagnosis, complete disease census or operational early-warning service.**